# MeAJOR DistilBERT Baseline

This notebook fine-tunes a pre-trained DistilBERT model for binary phishing email classification on the MeAJOR dataset. The resulting metrics are stored in the shared results CSV so they can be compared directly against the classical and deep learning baselines.

## 1. Import Libraries

All libraries needed for data loading, tokenisation, model training, evaluation, and results persistence are imported here.

In [1]:
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# HuggingFace Transformers
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# Sklearn metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

# Visualisation
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Reproducibility and Configuration

All random seeds are fixed at `42` and every hyper-parameter is defined as an `ALL_CAPS` constant so that results can be reproduced and adjusted easily.

In [2]:
# Configuration constants
SEED          = 42
MODEL_NAME    = "distilbert-base-uncased"
MAX_LEN       = 256
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 5
WARMUP_RATIO  = 0.1
WEIGHT_DECAY  = 1e-2
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

data_dir    = Path("../data/processed/meajor/")
results_dir = Path("../results/")
results_dir.mkdir(parents=True, exist_ok=True)

# Fix all random seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")

Device: mps


## 3. Load Processed Data

The 60% train and 40% test Parquet files produced upstream are loaded here. These contain the necessary `text` and `label` columns.

In [3]:
# Load the 60% training split
train_df = pd.read_parquet(data_dir / "meajor_train_60.parquet")

# Load the 40% test split
test_df = pd.read_parquet(data_dir / "meajor_test_40.parquet")

print(f"Train size: {len(train_df)} rows")
print(f"Test size:  {len(test_df)} rows")
print(f"Columns: {train_df.columns.tolist()}")

Train size: 62743 rows
Test size:  41829 rows
Columns: ['text', 'label']


## 4. Verify Loaded Data

A quick check to review data structure and confirm the label distribution is preserved across the train/test splits before processing.

In [4]:
# Preview the first 3 rows to confirm structure looks correct
print("Data Preview:")
print(train_df[["text", "label"]].head(3))
print()

# Confirm label balance matches the full dataset
print("Train label distribution:")
labels_train = train_df["label"].value_counts()
props_train = train_df["label"].value_counts(normalize=True)
for label, count in labels_train.items():
    print(f"{label}: {count} ({props_train[label]:.4f})")
print()
print("Test label distribution:")
labels_test = test_df["label"].value_counts()
props_test = test_df["label"].value_counts(normalize=True)
for label, count in labels_test.items():
    print(f"{label}: {count} ({props_test[label]:.4f})")

Data Preview:
                                                text  label
0  Subject: OVER [FINANCIAL_INFO] Body: HAS ANY O...      0
1  Subject: Or each ellenton Body: THE ALERT IS O...      1
2  Subject: Best place to find cure for your dise...      1

Train label distribution:
0: 34768 (0.5541)
1: 27975 (0.4459)

Test label distribution:
0: 23179 (0.5541)
1: 18650 (0.4459)


## 5. Tokenisation

The DistilBERT tokenizer is loaded from pre-trained weights and truncates to `MAX_LEN=256` tokens. Padding guarantees uniform lengths. The tokenizer is strictly not fitted on the MeAJOR dataset.

In [5]:
# Load DistilBERT tokenizer from pretrained weights — no fitting on local data
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

# Sanity check: tokenise one example to confirm truncation and padding
sample = tokenizer(
    train_df["text"].iloc[0],
    truncation=True,
    max_length=MAX_LEN,
    padding="max_length",
    return_tensors="pt",
)
print(f"Sample input_ids shape    : {sample['input_ids'].shape}")
print(f"Sample attention_mask shape: {sample['attention_mask'].shape}")

Sample input_ids shape    : torch.Size([1, 256])
Sample attention_mask shape: torch.Size([1, 256])


## 6. PyTorch Dataset and DataLoaders

The custom `EmailDataset` tokenises emails dynamically. We build a random 10% validation set independently from the train set before injecting all splits into the PyTorch DataLoaders.

In [6]:
class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        return (
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )


# Hold out a random 10% of training data as a validation set using specific SEED
val_df       = train_df.sample(frac=0.1, random_state=SEED)
train_sub_df = train_df.drop(val_df.index).reset_index(drop=True)
val_df       = val_df.reset_index(drop=True)

train_dataset = EmailDataset(
    train_sub_df["text"].tolist(), train_sub_df["label"].tolist(), tokenizer, MAX_LEN
)
val_dataset = EmailDataset(
    val_df["text"].tolist(), val_df["label"].tolist(), tokenizer, MAX_LEN
)
test_dataset = EmailDataset(
    test_df["text"].tolist(), test_df["label"].tolist(), tokenizer, MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

Train batches : 3530
Val   batches : 393
Test  batches : 2615


## 7. Model Definition

The pre-trained `distilbert-base-uncased` checkpoints initialize the weights. The head is configured to map into 2 distinct output labels.

> **Why DistilBERT?** DistilBERT is a distilled version of BERT that is smaller and faster while retaining most of BERT's downstream performance, making it a good fit for comparing transformer accuracy against inference efficiency in this project.

In [7]:
# Load DistilBERT with a two-class classification head
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {total_params:,}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total trainable parameters: 66,955,010


## 8. Training and Evaluation Functions

Functions abstract away the PyTorch boilerplate, handling the training steps, metric calculation including FPR from the confusion matrix, and robust, DataLoader-free inference timing.

In [ ]:
from tqdm.auto import tqdm

def train_epoch(model, loader, optimiser, scheduler, device):
    """Run one full training pass over the loader and return average loss."""
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    for input_ids, attention_mask, labels in tqdm(loader, desc="Training", leave=False):
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels         = labels.to(device)

        optimiser.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)
        loss.backward()
        optimiser.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, device):
    """Evaluate the model over the loader; return loss and classification metrics."""
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    all_labels, all_preds = [], []

    with torch.no_grad():
        for input_ids, attention_mask, labels in tqdm(loader, desc="Evaluating", leave=False):
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels         = labels.to(device)

            outputs    = model(input_ids=input_ids, attention_mask=attention_mask)
            loss       = criterion(outputs.logits, labels)
            total_loss += loss.item()

            preds = outputs.logits.argmax(dim=1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    prec     = precision_score(all_labels, all_preds, zero_division=0)
    rec      = recall_score(all_labels, all_preds, zero_division=0)
    f1       = f1_score(all_labels, all_preds, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return avg_loss, acc, prec, rec, f1, fpr, all_labels, all_preds


def measure_inference_time(model, loader, device, n_runs=3):
    """Measure model-only inference time in seconds, excluding DataLoader overhead.
    Pre-converts the test set into a list of fixed-size batches before timing."""
    from tqdm.auto import tqdm
    import time
    import numpy as np
    import torch
    
    model.eval()
    # Pre-collect all batches into lists to exclude DataLoader latency
    batches_ids = []
    batches_masks = []
    total_emails = 0
    
    for input_ids, attention_mask, _ in tqdm(loader, desc="Stacking inputs", leave=False):
        batches_ids.append(input_ids.to(device))
        batches_masks.append(attention_mask.to(device))
        total_emails += input_ids.size(0)
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            # Time the forward passes on the pre-loaded batches
            start = time.perf_counter()
            for b_ids, b_masks in zip(batches_ids, batches_masks):
                _ = model(input_ids=b_ids, attention_mask=b_masks)
            end = time.perf_counter()
            times.append(end - start) # stored directly in seconds
    avg_total_sec     = np.mean(times)
    avg_per_email_sec = avg_total_sec / total_emails
    return round(avg_total_sec, 4), round(avg_per_email_sec, 6)

## 9. Train Model

The model is fully fine-tuned with `AdamW` and a linear warmup scheduler. A neat text table continuously outputs the epoch evaluation tracking standard performance items.

In [9]:
# Instantiate AdamW optimiser with specific weight decay
optimiser = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Linear warmup scheduler
total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimiser,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# History tracked for plotting loops
train_losses, val_losses = [], []
val_accuracies, val_f1_scores = [], []

print("=" * 64)
print(
    f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>8} | "
    f"{'Val Acc':>7} | {'Val F1':>6} | {'Val FPR':>7}"
)
print("-" * 64)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(
        model, train_loader, optimiser, scheduler, DEVICE
    )
    val_loss, val_acc, _, _, val_f1, val_fpr, _, _ = evaluate(
        model, val_loader, DEVICE
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_f1_scores.append(val_f1)

    print(
        f"{epoch:>5} | {train_loss:>10.4f} | {val_loss:>8.4f} | "
        f"{val_acc:>7.4f} | {val_f1:>6.4f} | {val_fpr:>7.4f}"
    )

print("-" * 64)

Epoch | Train Loss | Val Loss | Val Acc | Val F1 | Val FPR
----------------------------------------------------------------


    1 |     0.1210 |   0.0525 |  0.9806 | 0.9786 |  0.0286


    2 |     0.0294 |   0.0398 |  0.9863 | 0.9848 |  0.0165


    3 |     0.0122 |   0.0471 |  0.9872 | 0.9856 |  0.0035


    4 |     0.0042 |   0.0452 |  0.9892 | 0.9879 |  0.0078


    5 |     0.0015 |   0.0513 |  0.9893 | 0.9881 |  0.0075
----------------------------------------------------------------


## 10. Final Test Evaluation

Final scoring applies the trained model on entirely unseen test rows to evaluate generalisation. A structured precision/recall breakdown alongside inference latency forms the required metrics block.

In [10]:
test_loss, test_acc, test_prec, test_rec, test_f1, test_fpr, true_labels, pred_labels = evaluate(
    model, test_loader, DEVICE
)
test_inf_total, test_inf_per_email = measure_inference_time(model, test_loader, DEVICE)

# Formatted Block
print("=" * 55)
print("  Final Test Results \u2014 DistilBERT")
print("=" * 55)
print(f"  Accuracy        : {test_acc:.4f}")
print(f"  Precision       : {test_prec:.4f}")
print(f"  Recall          : {test_rec:.4f}")
print(f"  F1 Score        : {test_f1:.4f}")
print(f"  False Pos. Rate : {test_fpr:.4f}")
print(f"  Inference Time  : {test_inf_per_email * 1000:.4f} ms/email")  # Block formats visually as ms
print()

print("Classification Report:")
print(classification_report(true_labels, pred_labels, target_names=["Legitimate (0)", "Phishing (1)"]))

# Build the results dictionary matching the requested schema
results_dict = {
    "dataset":                      "MeAJOR",
    "model":                        "DistilBERT",
    "accuracy":                     round(test_acc,  4),
    "precision":                    round(test_prec, 4),
    "recall":                       round(test_rec,  4),
    "f1":                           round(test_f1,   4),
    "false_positive_rate":          round(test_fpr,  4),
    "inference_time_total_sec":     test_inf_total,
    "inference_time_per_email_sec": test_inf_per_email,
    "train_rows":                   62743,
    "test_rows":                    41829,
    "vocab_size":                   tokenizer.vocab_size,
}

# Display a one-row pandas DataFrame output naturally at the end
pd.DataFrame([results_dict])

RuntimeError: Invalid buffer size: 30.64 GB

## 11. Confusion Matrix

Plotting the matrix of True Positives versus False Negatives grants an immediate visual gauge on bias and errors.

In [ ]:
figures_dir = results_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

cm = confusion_matrix(true_labels, pred_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legitimate", "Phishing"])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False)
ax.set_title("Confusion Matrix \u2014 DistilBERT")
plt.tight_layout()
plt.savefig(figures_dir / "meajor_cm_distilbert.png", dpi=150)
plt.show()

## 12. Training Curves

Training and validation loss are plotted to explicitly verify when convergence stabilizes and to track early signs of over-fitting.

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_losses, label="Train Loss", marker="o")
axes[0].plot(epochs, val_losses,   label="Val Loss",   marker="o")
axes[0].set_title("DistilBERT \u2014 Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs, val_f1_scores, label="Val F1", marker="s", color="green")
axes[1].set_title("DistilBERT \u2014 Val F1")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(figures_dir / "meajor_distilbert_training_curves.png", dpi=150)
plt.show()

## 13. Save Results

The metrics are appended directly into structured persistent storage, ensuring cross-comparison schemas are maintained.

In [ ]:
metrics_dir = results_dir / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

save_path  = metrics_dir / "meajor_distilbert_results.csv"
results_df = pd.DataFrame([results_dict])

# Save precisely with index=False using standard dataframe to CSV method
results_df.to_csv(save_path, index=False)

print(f"Results saved to {save_path}")